In [1]:
import os
import sys

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

import torch
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from torch.utils.data import DataLoader

from Training.emotion_dataset import EmotionDataset

from Models.multimodal_model import MultimodalEmotionModel

In [2]:
fusion_df = pd.read_csv(
    "../Outputs/multimodal_features.csv"
)

label_encoder = LabelEncoder()

fusion_df["emotion"] = label_encoder.fit_transform(
    fusion_df["emotion"]
)

psycho_features = fusion_df.iloc[:,2:36].values

hubert_features = fusion_df.iloc[:,36:].values

labels = fusion_df["emotion"].values

print(fusion_df.shape)

(2880, 804)


In [3]:
X_psy_train, X_psy_test, \
X_hubert_train, X_hubert_test, \
y_train, y_test = train_test_split(

    psycho_features,

    hubert_features,

    labels,

    test_size=0.2,

    random_state=42,

    stratify=labels

)

In [4]:
train_dataset = EmotionDataset(

    X_psy_train,

    X_hubert_train,

    y_train

)

train_loader = DataLoader(

    train_dataset,

    batch_size=32,

    shuffle=False
)

In [5]:
device = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else

    "cpu"

)

print(device)

cpu


In [6]:
model = MultimodalEmotionModel(

    psycho_dim=34,

    hubert_dim=768,

    hidden_dim=128,

    num_classes=8

)

model.load_state_dict(

    torch.load(

        "../Models/best_model.pth",

        map_location=device

    )

)

model = model.to(device)

model.eval()

print("Best Model Loaded")

Best Model Loaded


In [7]:
embeddings = []

emotion_labels = []

predicted_labels = []

model.eval()

with torch.no_grad():

    for psycho_batch, hubert_batch, labels_batch in train_loader:

        psycho_batch = psycho_batch.to(device)

        hubert_batch = hubert_batch.to(device)

        labels_batch = labels_batch.to(device)

        logits, fused_embedding, attention = model(

            psycho_batch,

            hubert_batch

        )

        predictions = torch.argmax(

            logits,

            dim=1

        )

        embeddings.extend(

            fused_embedding.cpu().numpy()

        )

        emotion_labels.extend(

            labels_batch.cpu().numpy()

        )

        predicted_labels.extend(

            predictions.cpu().numpy()

        )

print("Embedding Extraction Completed")

Embedding Extraction Completed


In [8]:
embeddings = np.array(embeddings)

emotion_labels = np.array(emotion_labels)

predicted_labels = np.array(predicted_labels)

print("Embeddings :", embeddings.shape)

print("Emotion Labels :", emotion_labels.shape)

print("Predicted Labels :", predicted_labels.shape)

Embeddings : (2304, 128)
Emotion Labels : (2304,)
Predicted Labels : (2304,)


In [9]:
embedding_df = pd.DataFrame(

    embeddings,

    columns=[

        f"embedding_{i}"

        for i in range(128)

    ]

)

embedding_df["true_emotion"] = emotion_labels

embedding_df["predicted_emotion"] = predicted_labels

embedding_df.head()

,embedding_0,embedding_1,embedding_2,embedding_3,embedding_4,embedding_5,embedding_6,embedding_7,embedding_8,embedding_9,...,embedding_120,embedding_121,embedding_122,embedding_123,embedding_124,embedding_125,embedding_126,embedding_127,true_emotion,predicted_emotion
0,0.245598,4.410629,0.009100,-1.836642,0.201188,1.075351,3.277626,1.392569,3.430138,-7.240261,...,3.705180,-0.978586,0.265383,1.166758,1.703877,-0.889544,-3.424323,3.727982,3,3
1,2.677036,-1.445399,-3.515369,-2.076035,0.689610,-0.342064,-1.764328,-3.316903,4.081441,-0.467389,...,1.224560,-4.000584,1.090191,-0.071929,0.584847,1.922777,0.426538,1.523122,4,4
2,2.110893,-0.558119,-2.590115,-3.608394,-0.262093,-2.565998,0.202693,0.290771,0.841821,1.948659,...,-1.966511,-0.558435,-3.851785,-2.855894,0.454673,-0.022006,-2.287657,-1.269096,2,2
3,0.374477,5.338937,0.341897,-2.246597,-1.117289,3.616482,5.236578,4.115743,4.539064,-8.050626,...,3.589935,-3.679926,-0.135914,4.017133,2.712927,-0.961693,-6.079439,4.121679,3,3
4,-2.058469,-6.250026,3.106769,-4.009958,1.628585,0.762761,-2.327261,-3.697174,-2.606567,3.202949,...,-0.951866,6.617577,4.258047,-4.522090,-6.292509,0.682654,6.747602,-3.480375,0,0


In [10]:
embedding_df["true_emotion"] = label_encoder.inverse_transform(

    embedding_df["true_emotion"]

)

embedding_df["predicted_emotion"] = label_encoder.inverse_transform(

    embedding_df["predicted_emotion"]

)

embedding_df.head()

,embedding_0,embedding_1,embedding_2,embedding_3,embedding_4,embedding_5,embedding_6,embedding_7,embedding_8,embedding_9,...,embedding_120,embedding_121,embedding_122,embedding_123,embedding_124,embedding_125,embedding_126,embedding_127,true_emotion,predicted_emotion
0,0.245598,4.410629,0.009100,-1.836642,0.201188,1.075351,3.277626,1.392569,3.430138,-7.240261,...,3.705180,-0.978586,0.265383,1.166758,1.703877,-0.889544,-3.424323,3.727982,fearful,fearful
1,2.677036,-1.445399,-3.515369,-2.076035,0.689610,-0.342064,-1.764328,-3.316903,4.081441,-0.467389,...,1.224560,-4.000584,1.090191,-0.071929,0.584847,1.922777,0.426538,1.523122,happy,happy
2,2.110893,-0.558119,-2.590115,-3.608394,-0.262093,-2.565998,0.202693,0.290771,0.841821,1.948659,...,-1.966511,-0.558435,-3.851785,-2.855894,0.454673,-0.022006,-2.287657,-1.269096,disgust,disgust
3,0.374477,5.338937,0.341897,-2.246597,-1.117289,3.616482,5.236578,4.115743,4.539064,-8.050626,...,3.589935,-3.679926,-0.135914,4.017133,2.712927,-0.961693,-6.079439,4.121679,fearful,fearful
4,-2.058469,-6.250026,3.106769,-4.009958,1.628585,0.762761,-2.327261,-3.697174,-2.606567,3.202949,...,-0.951866,6.617577,4.258047,-4.522090,-6.292509,0.682654,6.747602,-3.480375,angry,angry


In [11]:
embedding_df.to_csv(

    "../Outputs/emotion_embeddings.csv",

    index=False

)

print("Emotion Embeddings Saved Successfully")

Emotion Embeddings Saved Successfully


In [12]:
saved_df = pd.read_csv(

    "../Outputs/emotion_embeddings.csv"

)

print(saved_df.shape)

saved_df.head()

(2304, 130)


,embedding_0,embedding_1,embedding_2,embedding_3,embedding_4,embedding_5,embedding_6,embedding_7,embedding_8,embedding_9,...,embedding_120,embedding_121,embedding_122,embedding_123,embedding_124,embedding_125,embedding_126,embedding_127,true_emotion,predicted_emotion
0,0.245598,4.410629,0.009100,-1.836642,0.201188,1.075351,3.277626,1.392569,3.430138,-7.240261,...,3.705180,-0.978586,0.265383,1.166758,1.703877,-0.889544,-3.424323,3.727982,fearful,fearful
1,2.677036,-1.445399,-3.515369,-2.076035,0.689610,-0.342064,-1.764328,-3.316903,4.081441,-0.467389,...,1.224560,-4.000584,1.090191,-0.071929,0.584847,1.922777,0.426538,1.523122,happy,happy
2,2.110892,-0.558119,-2.590115,-3.608394,-0.262093,-2.565998,0.202693,0.290771,0.841821,1.948659,...,-1.966511,-0.558435,-3.851785,-2.855894,0.454673,-0.022006,-2.287657,-1.269095,disgust,disgust
3,0.374477,5.338937,0.341897,-2.246597,-1.117289,3.616483,5.236579,4.115743,4.539064,-8.050626,...,3.589935,-3.679926,-0.135914,4.017133,2.712927,-0.961693,-6.079439,4.121679,fearful,fearful
4,-2.058469,-6.250026,3.106769,-4.009958,1.628585,0.762761,-2.327261,-3.697174,-2.606567,3.202949,...,-0.951866,6.617577,4.258047,-4.522090,-6.292509,0.682654,6.747602,-3.480375,angry,angry


In [13]:
print(saved_df.shape)

(2304, 130)


In [14]:
from LLM.prompt_builder import PromptBuilder
import numpy as np

builder = PromptBuilder()

dummy_embedding = np.random.rand(128)

prompt = builder.build_prompt(
    "happy",
    dummy_embedding
)

print(prompt)


You are an emotionally intelligent AI assistant.

Detected Emotion:
happy

Emotion Embedding (first 10 values):
[0.006, 0.095, 0.344, 0.316, 0.344, 0.345, 0.342, 0.902, 0.842, 0.843]

Instructions:

1. Understand the emotional state of the speaker.
2. Respond empathetically.
3. Keep the information factually correct.
4. Avoid exaggeration.
5. Maintain a supportive and professional tone.

Response:

